# Polars, hands-on — *"When the DataFrame Broke"*
### Big Data Analytics · MBA (Business Analytics) · Session 2 · QuickKart

This notebook is the lab that goes with Session 2. It:

- **generates its own sample data** — nothing to download, no file to ship;
- **runs identically on macOS and Windows** (and Linux) — every path uses `pathlib`, every install targets the current interpreter;
- follows the same story as the lecture — filter out tiny orders, group by region, find the top region.

**How to run:** *Kernel → Restart & Run All* (Jupyter) or *Run All* (VS Code). The first cell installs Polars if it is missing.

> Cross-platform notes are called out like this whenever a line was written to behave the same on Mac and Windows.

---

### What each section covers

| Section | What it teaches |
|---|---|
| 0 · Setup | Install Polars into *this* interpreter, safely on every OS |
| 1 · Generate the sample data | Build a realistic, seasonal QuickKart order log — 6 regions, 10 categories, 3 months of dates |
| 1b · Seasonality check | Confirm the festive-sale skew we built in actually shows up |
| 2 · Eager vs. lazy | `read_csv` (eager) vs. `scan_csv` (lazy) — the core Polars concept |
| 3 · The business question | One real lazy pipeline: filter → group → aggregate → sort |
| 4 · Why lazy can be faster | `.explain()` — predicate pushdown and projection pushdown |
| 5 · Same logic in pandas | Syntax comparison, for students coming from pandas |
| 6 · Your turn | A fill-in-the-blanks practice exercise with a self-check solution |
| 7 · Recap | Summary + cross-platform checklist |


## 0 · Setup — make sure Polars is installed

We install into **the same Python that is running this notebook** using `sys.executable`.
That is the one line people get wrong: a bare `pip install` can target a *different* Python on
Windows than the kernel you are using. `sys.executable` avoids that on every OS.

**Why this matters:** on Windows especially, it's common to have several Pythons installed (one from
python.org, one bundled with Anaconda, one from the Microsoft Store). Running `pip install polars` in
a terminal can install into a *different* Python than the one Jupyter's kernel is using — so the import
still fails even though "pip said it worked." Calling `[sys.executable, "-m", "pip", "install", ...]`
sidesteps that entirely: it always installs into the exact interpreter that is running this notebook.

The `ensure()` helper below tries to `import` first and only installs if that fails, so re-running the
notebook doesn't reinstall Polars every single time.


In [ ]:
import importlib, subprocess, sys  # standard-library tools for checking/installing packages

def ensure(module_name, pip_name=None):
    "Import a package; if missing, install it into THIS interpreter (Mac/Win/Linux safe)."
    try:
        return importlib.import_module(module_name)  # try to load the package if it's already installed
    except ImportError:
        print(f"Installing {pip_name or module_name} ...")  # tell the user we're about to install it
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or module_name])  # install it into this interpreter
        return importlib.import_module(module_name)  # now that it's installed, load it

pl = ensure("polars")  # make sure Polars is installed, then load it as "pl"
print("Polars version:", pl.__version__)  # show which version of Polars we're using
print("Running on     :", sys.platform)   # 'darwin' = macOS, 'win32' = Windows, 'linux' = Linux


In [ ]:
import random  # standard-library module for generating random numbers
from pathlib import Path  # standard-library module for building file paths safely

# Cross-platform folder + file paths. Path() joins with the RIGHT separator
# on every OS ('/' on Mac, '\\' on Windows) — never hard-code slashes.
DATA_DIR = Path.cwd() / "quickkart_data"  # folder where we'll store our generated data
DATA_DIR.mkdir(exist_ok=True)  # create that folder if it doesn't already exist
CSV_PATH = DATA_DIR / "quickkart_orders.csv"  # full path to the CSV file we will create

print("Data folder:", DATA_DIR)  # show the folder path
print("CSV file   :", CSV_PATH)  # show the CSV file path


In [ ]:
import datetime  # standard-library module for representing calendar dates

random.seed(42)   # fix the "random" starting point so everyone gets the same data on every machine

REGIONS = [  # India's 6 official zonal council divisions — more realistic than a generic 5-way split
    "North", "South", "East", "West", "Central", "Northeast",
]
CATEGORIES = [  # 10 categories — a broader, more realistic QuickKart catalog
    "Groceries", "Electronics", "Apparel", "Pharmacy", "Home",
    "Beauty & Personal Care", "Toys & Games", "Sports & Fitness",
    "Stationery & Books", "Pet Supplies",
]
N = 50_000  # how many fake orders we want to generate

# --- Date range + festive-sale window, for seasonality ---
START_DATE = datetime.date(2025, 9, 1)     # data starts here
END_DATE = datetime.date(2025, 11, 30)     # data ends here
FESTIVE_START = datetime.date(2025, 10, 15)  # Diwali festive-sale window starts here
FESTIVE_END = datetime.date(2025, 11, 5)     # ...and ends here
TOTAL_DAYS = (END_DATE - START_DATE).days      # how many days the full range spans
FESTIVE_DAYS = (FESTIVE_END - FESTIVE_START).days  # how many days the festive window spans

def random_order_date():
    "Pick a random date, deliberately skewed toward the festive-sale window (seasonality)."
    if random.random() < 0.45:   # 45% of orders land inside the 3-week festive window
        offset = random.randint(0, FESTIVE_DAYS)
        return FESTIVE_START + datetime.timedelta(days=offset)
    offset = random.randint(0, TOTAL_DAYS)   # the rest spread across the full 3 months
    return START_DATE + datetime.timedelta(days=offset)

rows = []  # empty list that will hold one entry per order
for order_id in range(1, N + 1):        # repeat once for every order, numbered 1 to 50,000
    region   = random.choice(REGIONS)     # pick one region at random for this order
    category = random.choice(CATEGORIES)  # pick one category at random for this order
    order_date = random_order_date()      # pick a date, biased toward the festive-sale window
    # ~30% tiny orders below 500, the rest larger — so the >500 filter actually does something
    if random.random() < 0.30:  # 30% of the time, make this a small order
        order_value = round(random.uniform(50, 500), 2)    # random small order value, rounded to 2 decimals
    else:  # the other 70% of the time
        order_value = round(random.uniform(500, 8000), 2)  # random larger order value, rounded to 2 decimals
    units = random.randint(1, 5)  # random number of items bought, between 1 and 5
    rows.append((order_id, region, category, order_value, units, order_date.isoformat()))  # save this order as one row of data

orders = pl.DataFrame(  # build a Polars table (DataFrame) out of all the rows we generated
    rows,
    schema=["OrderID", "Region", "Category", "OrderValue", "Units", "OrderDate"],  # name each column
    orient="row",  # tell Polars our data is a list of rows (not columns)
)

# write_csv accepts a pathlib.Path directly and picks the OS-correct path — cross-platform.
orders.write_csv(CSV_PATH)  # save the table to a CSV file on disk
print(f"Wrote {orders.height:,} rows to {CSV_PATH.name}")  # confirm how many rows were written
print(f"Regions: {len(REGIONS)}  |  Categories: {len(CATEGORIES)}  |  Date range: {START_DATE} to {END_DATE}")
orders.head()  # preview the first few rows of the table


In [ ]:
monthly = (
    pl.scan_csv(CSV_PATH)                                         # lazy: nothing runs yet
      .with_columns(pl.col("OrderDate").str.slice(0, 7).alias("Month"))  # "YYYY-MM-DD" -> "YYYY-MM"
      .group_by("Month")                                           # one group per calendar month
      .agg(
          pl.len().alias("NumberOfOrders"),                        # how many orders that month
          pl.col("OrderValue").sum().round(2).alias("TotalSales"),  # total sales that month
      )
      .sort("Month")                                                # chronological order
      .collect()
)
monthly


In [ ]:
eager = pl.read_csv(CSV_PATH)          # runs immediately — reads the whole file into memory now (eager)
lazy  = pl.scan_csv(CSV_PATH)          # builds a plan only — reads nothing yet (lazy)

print("read_csv  ->", type(eager).__name__)   # DataFrame  (eager)
print("scan_csv  ->", type(lazy).__name__)     # LazyFrame  (lazy)


In [ ]:
query = (
    pl.scan_csv(CSV_PATH)                                   # lazy: nothing runs yet, just opens a plan to read the CSV
      .filter(pl.col("OrderValue") > 500)                   # step 1: drop tiny orders (500 or below)
      .group_by("Region")                                   # step 2: one group per region
      .agg(                                                 # step 3: compute these numbers for each group
          pl.len().alias("NumberOfOrders"),                 #   - how many orders in this region
          pl.col("OrderValue").sum().round(2).alias("TotalSales"),        #   - total sales in this region
          pl.col("OrderValue").mean().round(2).alias("AverageOrderValue"),#   - average order value in this region
      )
      .sort("TotalSales", descending=True)                  # step 4: sort regions, highest total sales first
)

result = query.collect()    # <-- THIS line actually runs the whole pipeline and gives us a real table
result   # display the resulting table


In [ ]:
top = result.row(0, named=True)  # grab the first row (the top region) as a named, dictionary-like object
print(f"Top region: {top['Region']}  "  # print a friendly summary sentence built from that row's values
      f"(TotalSales = {top['TotalSales']:,.2f} INR across {top['NumberOfOrders']:,} orders)")


In [ ]:
plan = (
    pl.scan_csv(CSV_PATH)                                    # start the same lazy plan as before
      .filter(pl.col("OrderValue") > 500)                    # keep only orders above 500
      .group_by("Region")                                    # group the remaining orders by region
      .agg(pl.col("OrderValue").sum().alias("TotalSales"))    # add up OrderValue per region
)
print(plan.explain())     # print the optimized query plan (read it bottom-up) — nothing is executed yet


In [ ]:
try:
    import pandas as pd                               # try to load pandas, a different (older, eager-only) data library
    pdf = pd.read_csv(CSV_PATH)                       # eager: whole file into memory now
    big = pdf[pdf["OrderValue"] > 500]                # keep only rows where OrderValue is above 500
    summary = (big.groupby("Region")["OrderValue"]    # group the filtered rows by region
                  .agg(["count", "sum", "mean"])        # compute count, total, and average per region
                  .sort_values("sum", ascending=False)   # sort so the highest total is first
                  .round(2))                             # round the numbers to 2 decimal places
    print("pandas result (same question):")   # label the output so it's clear what follows
    print(summary)   # show the pandas result
except ImportError:
    print("pandas not installed — skipping the comparison (Polars result above is enough).")  # skip gracefully if pandas is missing


## 6 · Your turn — fill in the blanks

Rebuild the same answer from scratch. Replace each `____` and run the cell.
The logic is the one you already know: **filter → group → aggregate → sort**.

The next cell is intentionally commented out (so "Run All" doesn't error on the blanks) — uncomment it,
fill in the four blanks, and run it yourself before checking the model solution below.


In [ ]:
# result_practice = (
#     pl.scan_csv(CSV_PATH)
#       .filter( ____ )          # keep OrderValue above 500
#       .group_by( ____ )        # group by Region
#       .agg( ____ )             # total sales per region
#       .sort( ____, descending=True )
# )
# print(result_practice.collect())


## 7 · Recap

- `pl.read_csv` is **eager**; `pl.scan_csv` is **lazy** — the word *scan* is the tell.
- A lazy pipeline runs only at the **trigger**: `.collect()` (or `.sink_csv()` to write straight to disk).
- Laziness lets the optimizer do **predicate pushdown** and **projection pushdown** — less data read, faster answer.
- The verbs never changed from pandas: **filter → group → aggregate → sort**. Only the engine did.
- Real-world data has **seasonality** (the festive-sale skew we built into `OrderDate`) — always sanity-check
  that a data-generation assumption actually shows up before building analysis on top of it (section 1b).

**Cross-platform checklist used in this notebook:** `sys.executable` for installs · `pathlib.Path`
for every path · standard-library `random`/`datetime` for data · `write_csv`/`scan_csv` take `Path` objects directly.
Nothing here is Mac-only or Windows-only.


In [ ]:
# Optional: clean up the generated file (safe on every OS via pathlib)
# CSV_PATH.unlink(missing_ok=True)      # delete the CSV
# DATA_DIR.rmdir()                       # remove the folder if now empty
print("Done. Generated data is in:", DATA_DIR)   # final message showing where the generated data lives
